In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

In [4]:
PREVIOUS_APPLICATION_DATA_PATH = "../datasets/raw/previous_application.csv"
IMAGES_PATH = "../images/"
OUTPUT_PATH = "../datasets/preprocess/"
TRAIN_DATA_PATH = "../datasets/raw/application_train.csv"

In [5]:
app_train = pd.read_csv(TRAIN_DATA_PATH)

previous_app = pd.read_csv(PREVIOUS_APPLICATION_DATA_PATH)

display(previous_app.head())
print("\n")
print(previous_app.info())

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  object 
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  object 
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  object 
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-

In [6]:
previous_app_agg = previous_app.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',                        # cuántas solicitudes previas tiene
    'AMT_CREDIT': ['mean', 'max', 'sum'],          # monto de crédito solicitado
    'AMT_ANNUITY': ['mean', 'max'],                # pago mensual (anualidad)
    'CNT_PAYMENT': ['mean', 'max'],                # plazo de pago en meses
    'DAYS_DECISION': ['mean', 'min'],               # qué tan reciente/antigua es la última decisión
})

previous_app_agg.columns = ['_'.join(col).upper() for col in previous_app_agg.columns]
previous_app_agg = previous_app_agg.reset_index()

previous_app_agg.head()

,SK_ID_CURR,SK_ID_PREV_COUNT,AMT_CREDIT_MEAN,AMT_CREDIT_MAX,AMT_CREDIT_SUM,AMT_ANNUITY_MEAN,AMT_ANNUITY_MAX,CNT_PAYMENT_MEAN,CNT_PAYMENT_MAX,DAYS_DECISION_MEAN,DAYS_DECISION_MIN
0,100001,1,23787.00,23787.0,23787.0,3951.000,3951.000,8.0,8.0,-1740.0,-1740
1,100002,1,179055.00,179055.0,179055.0,9251.775,9251.775,24.0,24.0,-606.0,-606
2,100003,3,484191.00,1035882.0,1452573.0,56553.990,98356.995,10.0,12.0,-1305.0,-2341
3,100004,1,20106.00,20106.0,20106.0,5357.250,5357.250,4.0,4.0,-815.0,-815
4,100005,2,20076.75,40153.5,40153.5,4813.200,4813.200,12.0,12.0,-536.0,-757


In [7]:
app_train = app_train.merge(previous_app_agg,on='SK_ID_CURR',how='left')

display(app_train.head())

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,SK_ID_PREV_COUNT,AMT_CREDIT_MEAN,AMT_CREDIT_MAX,AMT_CREDIT_SUM,AMT_ANNUITY_MEAN,AMT_ANNUITY_MAX,CNT_PAYMENT_MEAN,CNT_PAYMENT_MAX,DAYS_DECISION_MEAN,DAYS_DECISION_MIN
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,1.0,179055.00,179055.0,179055.0,9251.775,9251.775,24.000000,24.0,-606.000000,-606.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,3.0,484191.00,1035882.0,1452573.0,56553.990,98356.995,10.000000,12.0,-1305.000000,-2341.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,1.0,20106.00,20106.0,20106.0,5357.250,5357.250,4.000000,4.0,-815.000000,-815.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,9.0,291695.50,906615.0,2625259.5,23651.175,39954.510,23.000000,48.0,-272.444444,-617.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,6.0,166638.75,284400.0,999832.5,12278.805,22678.785,20.666667,48.0,-1222.833333,-2357.0
